In [4]:
from common.aws_config import aws_env_setup
from common.bedrock_config import get_anthropic_client
import json 

aws_env_setup()
client, model = get_anthropic_client()

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #"temperature": temperature
        "extra_body": {"temperature": temperature}
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

3 Functions -->

    a. run_prompt Function : prompt + test case , return o/p
    b. run_grade_case : Grades 
    c. run_eval : Coordinates the entire evaluation process

In [6]:
# run_prompt funtion 
def run_prompt(test_case):
    "Merges Prompt + test case input - Returns result"
    prompt = f"""
             Solve the following task:
             {test_case["task"]}
             """ 
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages, temperature=0.7)
    return output


In [7]:
def run_grade_case (test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    grade = 10 
    return {"input": test_case, "output": output, "grade": grade} #Dict 


In [8]:
def run_eval(dataset):
    """Loads dataset calls grading function """
    results = []
    for test_case in dataset:
        result = run_grade_case(test_case)
        results.append(result)
    return results


In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    #print(dataset)
results = run_eval(dataset)


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "input": {
      "task": "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., 's3://my-bucket-us-west-2/path/to/file'). The function should return the region code or None if not found."
    },
    "output": "# AWS Region Extraction from S3 URI\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_uri(s3_uri: str) -> Optional[str]:\n    \"\"\"\n    Extract AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/path' or 's3://bucket-name'\n    \n    Returns:\n        Region code (e.g., 'us-west-2') or None if not found\n    \n    Examples:\n        >>> extract_region_from_s3_uri('s3://my-bucket-us-west-2/path/to/file')\n        'us-west-2'\n        >>> extract_region_from_s3_uri('s3://bucket-eu-central-1')\n        'eu-central-1'\n        >>> extract_region_from_s3_uri('s3://my-bucket/path')\n        None\n    \"